# Project Track 5 - Rarefied/Kinetic Cavity AI (Advanced)

**Choose one variant:**

- **5A Noise-aware learning:** compare a surrogate trained on one stochastic seed with one trained on seed-averaged labels.
- **5B Generalization across Knudsen number:** use case-wise splitting and test at an unseen higher Kn.

This track uses a CPU-friendly teaching DSMC-style solver. It is suitable for studying workflow and uncertainty; it is not a validation-quality production DSMC calculation. Version 1.2 pools raw moments over the full sampling window before computing temperature, so seed averaging targets random label noise rather than a hidden small-sample temperature bias.

## Required files
`P5_Rarefied_Cavity.ipynb`, `mini_dsmc.py`, `w5_common.py`, `w4utils.py`

<!-- MIE690A enriched learner edition v2 -->

## How to learn from this notebook

This is a guided computational laboratory, not a script to execute without reading. For every numbered stage:

1. read the physical question and write a prediction;
2. inspect the inputs, outputs, units, and split before running code;
3. run the cell and check assertions/warnings;
4. compare with the stated baseline or physical diagnostic; and
5. write one or two sentences explaining what the result does **and does not** establish.

Use **Restart and Run All** before treating any output as final. Hidden state from out-of-order execution is a reproducibility failure.

### Evidence contract

Keep four kinds of evidence separate:

- **numerical evidence:** residuals, accepted cases, data hashes, grid/time/particle budgets;
- **statistical evidence:** losses, relative errors, variability across seeds/cases;
- **physical evidence:** centerlines, walls, divergence, vortex structure, positivity, moments;
- **computational evidence:** runtime, memory, saved configuration, and machine-readable metrics.

A claim is only as strong as the weakest relevant layer.


## Learning objectives and scope warning

You will treat each `(Kn, Uwall, seed)` run as a complete stochastic physical case, quantify seed-to-seed label variability, compare single-seed and averaged labels or test unseen Knudsen number, and keep finite-window teaching evidence separate from validation-quality DSMC claims.

Prerequisites: Maxwellian sampling, DSMC move/collide/sample logic, Knudsen regimes, normalization, stochastic independence, and case-wise splitting.


In [ ]:
# Repository/Colab bootstrap. Run this before the import cell below.
from pathlib import Path
import sys

def _find_course_root(start=Path.cwd()):
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "common" / "w5_common.py").exists():
            return base
    # Colab flat-upload fallback: helper files and data beside the notebook.
    if (start / "w5_common.py").exists():
        return start
    raise FileNotFoundError(
        "Course root not found. Clone the repository, or upload w4utils.py, "
        "w5_common.py, the track-specific helpers, and cavity_data.npz as listed above."
    )

COURSE_ROOT = _find_course_root()
COMMON_DIR = COURSE_ROOT / "common" if (COURSE_ROOT / "common").exists() else COURSE_ROOT
DATASET_PATH = COURSE_ROOT / "data" / "cavity_data.npz"
if not DATASET_PATH.exists():
    DATASET_PATH = COURSE_ROOT / "cavity_data.npz"
sys.path.insert(0, str(COMMON_DIR))
print("Course root:", COURSE_ROOT)
print("Common helpers:", COMMON_DIR)
print("Dataset:", DATASET_PATH)


In [ ]:
from pathlib import Path
import time,importlib,json
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import tensorflow as tf
import mini_dsmc,w5_common
importlib.reload(mini_dsmc);importlib.reload(w5_common)
w5_common.set_global_seed(690)
VARIANT="5A"  # EDIT to 5B if approved.
CACHE=Path(f"P5_{VARIANT}_cases_v12.npz")
assert mini_dsmc.MINI_DSMC_VERSION == "1.2", "Upload mini_dsmc.py version 1.2 from this package."
print("mini_dsmc version:",mini_dsmc.MINI_DSMC_VERSION)


## What the mini solver can and cannot establish

The supplied solver preserves the pedagogical structure of particle transport, wall interaction, stochastic collisions, and macroscopic sampling. Its reduced grid, particles per cell, and averaging window make it suitable for workflow experiments, not for a new quantitative rarefied-cavity benchmark.

Knudsen number changes the ratio of molecular mean free path to cavity size. Changing `Kn` can change both the physical solution and the time required to reach a statistically stationary sampling window. A fixed number of steps is therefore not automatically a matched convergence standard across Kn.

**Budget audit:** record grid, particles per cell, time step, total steps, discarded transient, sample stride, wall model, and collision model for every case.


## 1. Generate or load complete physical cases

Every `(Kn,Uwall,seed)` combination is one physical simulation case. Grid points from one case must never be split randomly between train and test when the claim concerns new cases.

The revised defaults use 3200 steps and discard the first 2400. The number of stored field samples remains modest, but the averaging window is moved later in time to reduce transient contamination, especially when comparing different Knudsen numbers in Variant 5B. These remain finite-window teaching labels, not asymptotic validation data.

In [ ]:
if VARIANT=="5A":
    KNS=[0.05,0.10,0.20]; UWALLS=[0.4,0.7]; SEEDS=[11,22,33]
else:
    KNS=[0.05,0.10,0.20,0.40]; UWALLS=[0.4,0.7]; SEEDS=[11,22]

if not CACHE.exists():
    cases=[]
    for Kn in KNS:
        for Uw in UWALLS:
            for seed in SEEDS:
                cfg=mini_dsmc.MiniDSMCConfig(Kn=Kn,Uwall=Uw,seed=seed)
                print("running",Kn,Uw,seed)
                c=mini_dsmc.run_case(cfg)
                cases.append(c)
    payload={"Kn":np.array([c["config"]["Kn"] for c in cases]),
             "Uwall":np.array([c["config"]["Uwall"] for c in cases]),
             "seed":np.array([c["config"]["seed"] for c in cases]),
             "x":cases[0]["x"],"y":cases[0]["y"],
             "u":np.stack([c["u"] for c in cases]),"v":np.stack([c["v"] for c in cases]),
             "T":np.stack([c["T"] for c in cases]),"rho":np.stack([c["rho"] for c in cases]),
             "mini_dsmc_version":np.array(mini_dsmc.MINI_DSMC_VERSION)}
    np.savez_compressed(CACHE,**payload)
with np.load(CACHE) as z: D={k:z[k] for k in z.files}
print("cases",len(D["Kn"]),"field shape",D["u"].shape)


## Noisy labels and nondimensional targets

The model inputs use `log10(Kn)` because rarefaction spans orders of magnitude. Velocity is normalized by wall speed and temperature by wall temperature. Fit all statistical standardizers using development cases only.

Seed averaging reduces random variance approximately with the number of sufficiently independent realizations, but it does not remove systematic bias from cell size, time step, collision selection, wall model, or incomplete transient removal.


## 2. Assemble pointwise case data

The input is `(log10 Kn, Uwall, x, y)`. Outputs are `(u/Uwall, v/Uwall, T/Twall)`. Normalization statistics are fitted on development cases only.

Temperature labels are obtained from pooled particle moments over the whole sampling window with a Bessel correction. Averaging seeds can reduce random variance, but it cannot remove grid, time-step, collision-model, wall-model, or other shared systematic bias.

In [ ]:
def case_rows(dataset,indices):
    Xg,Yg=np.meshgrid(dataset["x"],dataset["y"]); X=[];Y=[]
    for i in indices:
        n=Xg.size; Uw=dataset["Uwall"][i]
        X.append(np.column_stack([np.full(n,np.log10(dataset["Kn"][i])),np.full(n,Uw),Xg.ravel(),Yg.ravel()]))
        Y.append(np.column_stack([(dataset["u"][i]/Uw).ravel(),(dataset["v"][i]/Uw).ravel(),dataset["T"][i].ravel()]))
    return np.vstack(X),np.vstack(Y)

def fit_model(Xtr,Ytr,Xva,Yva,seed=690):
    s=w5_common.fit_standardizers(Xtr,Ytr)
    m=w5_common.make_dense_model(4,3,(64,64,64),"tanh",seed)
    m.compile(tf.keras.optimizers.Adam(1e-3),loss="mse")
    h=m.fit(s.transform_x(Xtr),s.transform_y(Ytr),validation_data=(s.transform_x(Xva),s.transform_y(Yva)),
        epochs=750,batch_size=512,verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=60,restore_best_weights=True)])
    return {"model":m,"scalers":s,"history":h.history}

def predict_case(bundle,dataset,i):
    Xg,Yg=np.meshgrid(dataset["x"],dataset["y"]);n=Xg.size
    X=np.column_stack([np.full(n,np.log10(dataset["Kn"][i])),np.full(n,dataset["Uwall"][i]),Xg.ravel(),Yg.ravel()])
    P=bundle["scalers"].inverse_y(bundle["model"].predict(bundle["scalers"].transform_x(X),verbose=0))
    sh=Xg.shape;return P[:,0].reshape(sh),P[:,1].reshape(sh),P[:,2].reshape(sh)


## Variant 5A: define the independent comparison

The single-seed and seed-averaged models must be evaluated against a seed that contributed to neither training target. Compare model error with the seed-to-seed spread of the simulation labels. If model differences are smaller than label variability, the conclusion should be correspondingly cautious.


## 3A. Noise-aware single-seed versus seed-averaged labels

In [ ]:
if VARIANT=="5A":
    # Keep the original simulation dictionary unchanged.  Re-running this cell
    # is therefore safe and cannot append duplicate averaged cases.
    D_single={k:np.array(v,copy=True) for k,v in D.items()}
    train1=np.where(D_single["seed"]==11)[0]
    train2=np.where(D_single["seed"]==22)[0]
    test=np.where(D_single["seed"]==33)[0]  # independent stochastic realization

    avg_u=[];avg_v=[];avg_T=[];avg_rho=[];avg_Kn=[];avg_U=[]
    for i in train1:
        matches=train2[(D_single["Kn"][train2]==D_single["Kn"][i]) &
                       (D_single["Uwall"][train2]==D_single["Uwall"][i])]
        if len(matches)!=1:
            raise RuntimeError("Expected exactly one seed-22 match for every seed-11 condition")
        j=matches[0]
        avg_u.append(.5*(D_single["u"][i]+D_single["u"][j]))
        avg_v.append(.5*(D_single["v"][i]+D_single["v"][j]))
        avg_T.append(.5*(D_single["T"][i]+D_single["T"][j]))
        avg_rho.append(.5*(D_single["rho"][i]+D_single["rho"][j]))
        avg_Kn.append(D_single["Kn"][i]);avg_U.append(D_single["Uwall"][i])
    D_avg={"Kn":np.asarray(avg_Kn),"Uwall":np.asarray(avg_U),
           "seed":np.full(len(avg_Kn),-1),"x":D_single["x"],"y":D_single["y"],
           "u":np.stack(avg_u),"v":np.stack(avg_v),"T":np.stack(avg_T),
           "rho":np.stack(avg_rho)}
    assert np.sum(D_avg["seed"]==-1)==len(avg_Kn)

    def split_dev(dataset,indices):
        matches=indices[(dataset["Kn"][indices]==.2)&(dataset["Uwall"][indices]==.7)]
        if len(matches)!=1: raise RuntimeError("Validation condition must occur exactly once")
        val=matches[0]
        return indices[indices!=val],np.array([val])

    s_tr,s_va=split_dev(D_single,train1)
    avg_all=np.arange(len(D_avg["Kn"]))
    a_tr,a_va=split_dev(D_avg,avg_all)
    single=fit_model(*case_rows(D_single,s_tr),*case_rows(D_single,s_va),seed=690)
    averaged=fit_model(*case_rows(D_avg,a_tr),*case_rows(D_avg,a_va),seed=690)

    # Quantify the seed-to-seed label variability before comparing surrogates.
    seed_gap=[]
    for i,j in zip(train1,train2):
        du=(D_single["u"][i]-D_single["u"][j])/D_single["Uwall"][i]
        dv=(D_single["v"][i]-D_single["v"][j])/D_single["Uwall"][i]
        seed_gap.append(np.sqrt(np.mean(du*du+dv*dv)))
    print("Mean seed-11/22 velocity-label RMS gap:",float(np.mean(seed_gap)))


## Variant 5B: Knudsen generalization is regime generalization

Hold out complete Kn values, not random cells. An unseen higher Kn case can combine parameter extrapolation, altered non-equilibrium structure, and different label noise. Report whether the test is within the same qualitative regime or crosses from slip toward transition behavior.


## 3B. Generalization across Knudsen number

In [ ]:
if VARIANT=="5B":
    # Average the two seeds for each physical condition, then split by complete Kn cases.
    records=[]
    for Kn in KNS:
        for Uw in UWALLS:
            ids=np.where((D["Kn"]==Kn)&(D["Uwall"]==Uw))[0]
            records.append((Kn,Uw,D["u"][ids].mean(0),D["v"][ids].mean(0),D["T"][ids].mean(0)))
    D_kn={"Kn":np.array([r[0] for r in records]),"Uwall":np.array([r[1] for r in records]),
        "seed":np.full(len(records),-1),"x":D["x"],"y":D["y"],
        "u":np.stack([r[2] for r in records]),"v":np.stack([r[3] for r in records]),
        "T":np.stack([r[4] for r in records]),"rho":np.ones_like(np.stack([r[4] for r in records]))}
    train=np.where(np.isin(D_kn["Kn"],[.05,.10]))[0]
    val=np.where(D_kn["Kn"]==.20)[0]
    test=np.where(D_kn["Kn"]==.40)[0]
    model_kn=fit_model(*case_rows(D_kn,train),*case_rows(D_kn,val),seed=690)


## Physical checks for particle-derived fields

At minimum examine mass/normalization behavior, velocity boundary response, temperature positivity, centerlines, recirculation topology, and seed variability. Higher-order quantities normally converge more slowly than density and mean velocity. A smooth neural field can suppress visible noise while introducing bias; smoothness is not validation.


## 4. Evaluate against complete held-out cases and physical checks

In [ ]:
rows=[]; examples={}
if VARIANT=="5A":
    model_specs={"single-seed labels":(single,D_single),
                 "two-seed averaged labels":(averaged,D_single)}
    eval_ids=test
else:
    model_specs={"low-Kn trained model":(model_kn,D_kn)}
    eval_ids=test
for name,(b,eval_data) in model_specs.items():
    for i in eval_ids:
        up,vp,Tp=predict_case(b,eval_data,i); Uw=eval_data["Uwall"][i]
        tu,tv,tT=eval_data["u"][i]/Uw,eval_data["v"][i]/Uw,eval_data["T"][i]
        vel_rel=np.sqrt(np.sum((up-tu)**2+(vp-tv)**2))/(np.sqrt(np.sum(tu**2+tv**2))+1e-12)
        T_rel=np.linalg.norm(Tp-tT)/(np.linalg.norm(tT)+1e-12)
        top_mean=float(np.mean(up[-1])); bottom_min=float(np.min(up[:len(up)//2]))
        rows.append({"variant":VARIANT,"method":name,"Kn":eval_data["Kn"][i],"Uwall":Uw,
                     "seed":eval_data["seed"][i],"relative_L2_uv":vel_rel,"relative_L2_T":T_rel,
                     "min_temperature":float(Tp.min()),"mean_top_u/Uwall":top_mean,
                     "min_lower_u/Uwall":bottom_min})
        examples[(name,i)]=(up,vp,Tp)
results=pd.DataFrame(rows);results.to_csv("P5_results.csv",index=False);display(results)


In [ ]:
# Plot one complete held-out physical case.
eval_data=D_single if VARIANT=="5A" else D_kn
i=eval_ids[-1]; X,Y=np.meshgrid(eval_data["x"],eval_data["y"])
fig,ax=plt.subplots(len(model_specs)+1,3,figsize=(12,3.5*(len(model_specs)+1)))
ax=np.atleast_2d(ax)
Uw=eval_data["Uwall"][i]
ax[0,0].streamplot(X,Y,eval_data["u"][i]/Uw,eval_data["v"][i]/Uw,density=1.0);ax[0,0].set_title("held-out simulation")
a=ax[0,1].contourf(X,Y,eval_data["T"][i],25);fig.colorbar(a,ax=ax[0,1]);ax[0,1].set_title("simulation T")
ax[0,2].plot(eval_data["u"][i][:,len(eval_data["x"])//2]/Uw,eval_data["y"]);ax[0,2].set_title("simulation centerline")
for row,(name,(b,_)) in enumerate(model_specs.items(),1):
    up,vp,Tp=examples[(name,i)]
    ax[row,0].streamplot(X,Y,up,vp,density=1.0);ax[row,0].set_title(name)
    a=ax[row,1].contourf(X,Y,Tp,25);fig.colorbar(a,ax=ax[row,1]);ax[row,1].set_title("predicted T")
    ax[row,2].plot(eval_data["u"][i][:,len(eval_data["x"])//2]/Uw,eval_data["y"],"k",label="simulation")
    ax[row,2].plot(up[:,len(eval_data["x"])//2],eval_data["y"],"r--",label="model");ax[row,2].legend(fontsize=8)
plt.tight_layout();plt.show()


## Required report evidence

- State clearly that the supplied mini solver is a teaching model, not validation-quality DSMC.
- Keep complete simulation cases together in splits.
- Variant 5A: compare single-seed and seed-averaged labels against an independent seed; quantify seed-to-seed label variability.
- Variant 5B: report exactly which Kn values are train, validation, and blind test; explain regime extrapolation.
- Use at least two physical checks: positive temperature, correct lid-driven sign, recirculating lower-flow sign, or reasonable symmetry/trends.
- Identify whether error is dominated by surrogate approximation, stochastic labels, or out-of-range physics.
- Do not claim a smooth neural field proves the DSMC labels are converged.
- State explicitly that seed averaging reduces random sampling variance but does not remove shared discretization or model-form bias.
- Report the pooled temperature estimator and the revised simulation budget (`steps`, `sample_start`, and field-sample count).
- State that the labels are finite-window teaching data. For Variant 5B, discuss whether any remaining transient bias could vary with Kn and affect a regime-transfer claim.

## Optional stretch extensions (advanced / prize-track only)

Complete the required project first. With instructor approval, choose at most one:

1. Compare early and late sampling windows to quantify transient bias separately from seed-to-seed variance.
2. Estimate cellwise label variance from repeated seeds and use an inverse-variance weighted loss.
3. Build a compact shared-support representation of the kinetic cavity fields and benchmark fidelity versus storage against a simple grid or POD baseline.


## Concept check and further reading

1. Why is a new seed a different test from a new Knudsen number?
2. Which errors are reduced by averaging independent seeds?
3. Why can the same sampling window be less adequate at a different Kn?
4. Why should heat flux or stress be expected to converge more slowly than density?
5. What additional studies are required before making a production DSMC claim?

Read Bird (1994), Cercignani (1988), and the rarefied-flow sources in `references/README.md`.


## Reproducibility record

Before closing the notebook, record:

- Python and package versions;
- dataset hash and helper versions;
- every physical case in development, validation, and blind sets;
- every seed and candidate value tried;
- the selection rule and when it was frozen;
- output filenames and units; and
- any cell that was skipped, changed, or run with a reduced budget.

Restart the kernel and run all cells in order. If the result changes materially, report the variability instead of selecting the preferred run.

## Troubleshooting without corrupting the experiment

| Symptom | Safe action | Unsafe action |
| --- | --- | --- |
| Missing helper/data file | Re-run the bootstrap and verify paths/hash | Download an unlabeled older copy |
| Training is slow | Use the documented smoke configuration, then label it “smoke” | Quietly reduce epochs/data in the final claim |
| Validation is poor | Inspect scaling, split, and baseline; revise on development data | Open the blind case to choose settings |
| Blind result fails | Report/localize failure and propose a new future experiment | Tune on the blind case while keeping its label “blind” |
| Stochastic result changes | Run multiple declared seeds and report mean/spread | Keep rerunning until one result looks good |
| A neural model loses to interpolation | Verify fairness, then recommend the simpler method | Hide the baseline |

## Final report outline

1. **Question and hypothesis** — one falsifiable sentence.
2. **Data and split** — physical cases, numerical source, and blind unit.
3. **Baseline** — simplest credible comparator using the same allowed information.
4. **Modification** — the one controlled change.
5. **Selection** — validation-only candidates and frozen rule.
6. **Blind numerical result** — aggregate errors and variability.
7. **Physical result** — at least two diagnostics tied to the flow.
8. **Failure or limitation** — where confidence ends.
9. **Cost and reproducibility** — runtime, environment, seeds, saved files.
10. **Conclusion** — helped, hurt, or revealed a tradeoff; no forced positive AI claim.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Figures 10--12: standard DSMC versus hybrid neural--DSMC evidence.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../ARTICLE_FIGURE_MAP.md).
